### Data Dictionary:
* visitors: Average number of visitors, in millions, to the platform in the past week
* ad_impressions: Number of ad impressions, in millions, across all ad campaigns for the content (running and completed)
* major_sports_event: Any major sports event on the day
* genre: Genre of the content
* dayofweek: Day of the release of the content
* season: Season of the release of the content
* views_trailer: Number of views, in millions, of the content trailer
* views_content: Number of first-day views, in millions, of the content

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

In [ ]:
cData = pd.read_csv("ottdata.csv")

In [ ]:
cData.shape

In [ ]:
cData.head()

In [ ]:
cData.info()

In [ ]:
# let's check the statistical summary of the data
cData.describe()

## Checking for Duplicates

In [ ]:
cData.duplicated().sum()

* No duplicates in the dataset.

## Checking for Missing values

In [ ]:
cData.isnull().sum()

* No Null values in the data

In [ ]:
cData.describe().T

## Univariate Analysis

#### Let's check the distribution for numerical columns.

#### Observations on visitors

In [ ]:
sns.displot(data=cData,x='visitors',kind='kde')
plt.show()
sns.boxplot(data=cData,x='visitors')
plt.show()

* The distribution is normal
* There is one outlier present in this visitors column.
* Values above 2.2 million are being represented as outliers in the boxplot.

#### Observations on ad_impressions

In [ ]:
sns.displot(data=cData,x='ad_impressions',kind='kde')
plt.show()
sns.boxplot(data=cData,x='ad_impressions')
plt.show()

* The distribution is skewed towards the right.
* There is one outlier present in this ad_impressions column.

#### Observations on major_sports_event	

In [ ]:

sns.countplot(data=cData,x='major_sports_event')
plt.show()


* Data contains 40% Major Sports event data and 60% No Major Sports event data.

#### Observations on views_trailer

In [ ]:
sns.displot(data=cData,x='views_trailer',kind='kde')
plt.show()
sns.boxplot(data=cData,x='views_trailer')
plt.show()

* data is skewed towards right.
* There are many outliers in this attribute.

#### Observations on views_content

In [ ]:
sns.displot(data=cData,x='views_content',kind='kde')
plt.show()
sns.boxplot(data=cData,x='views_content')
plt.show()

* Data is slightly skewed towards right.
* There are some outliers in this attribute.
* Require Outlier treatment using IQR method. 

#### Observations on genre

In [ ]:
sns.countplot(data=cData,x='genre')
plt.xticks(rotation=90)
plt.show()

* All the genre's are having more or less equal count of content. But majority is categorized as others
* Thriller and Comedy Genre's have most content compared to other genre's.


#### Observations on dayofweek

In [ ]:
sns.countplot(data=cData,x='dayofweek')
plt.xticks(rotation=90)
plt.show()

* Most of the content released are on Friday and Wednesday.
* Monday and Tuesday has less number of contents released.

#### Observations on season

In [ ]:
sns.countplot(data=cData,x='season')
plt.xticks(rotation=90)
plt.show()

* Comparitatively speaking all the Seasons has euual amounts of content releases.
* Winter has most number of releases.

### Bivariate Analysis

In [ ]:
data_numeric = cData.select_dtypes('number')
plt.figure(figsize=(10,5))
sns.heatmap(data_numeric.corr(),annot=True,cmap='Spectral',vmin=-1,vmax=1)
plt.show()

* views_content shows high correlation with views_trailer. This indicates that people who sees trailer sees content on the same day.
* major_sports_event shows negative correlation with views_content. This indicates that on sports day content view on OTT is getting negatively impacted.
* views_content has positive correlation with visitors.
* ad_impressions has positive correlation but not that compared to other attributes in the data.

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=cData,x='views_trailer',y='views_content')
plt.show()

In [ ]:
sns.lmplot(data=cData,x='views_trailer',y='views_content',height=5,aspect=2)
plt.xlim(0,250)
plt.show()

* A positive correlation or an increasing trend can be clearly observed between views_trailer and views_content.

#### Relationship between views_content and day of release 

In [ ]:
Total_views = cData.groupby(['dayofweek'])[['views_content']].sum()
Total_views

In [ ]:
Total_views = Total_views.sort_values(by='views_content')

In [ ]:
sns.catplot(data=Total_views,x="dayofweek",y="views_content",hue="dayofweek",kind='bar')
plt.xticks(rotation=90)

* First day views_count is more for the content which release on wednessday and Friday.
* Content released on Friday has more total number of First day views.
* Content released on Wednesday is in next in line beside Friday wrt to First day total views.
* Content released on Tuesday has least total number of First day views compared to all other days. 

#### Relationship between views_content and season

In [ ]:
Total_views_season = cData.groupby(['season'])[['views_content']].sum()
Total_views_season

In [ ]:
Total_views_season = Total_views_season.sort_values(by='views_content')

In [ ]:
sns.catplot(data=Total_views_season,x="season",y="views_content",hue="season",kind='bar')
plt.xticks(rotation=90)

* Content released in Winter season has higher First day viewership
* Content released in Summer season is higher First day viewership next to winter.
* Content released in Fall season has least total First day viewrship compared to all the other seasons.

#### finding the percentage of outliers, in each column of the data, using IQR.

In [ ]:
# outlier detection using boxplot
# selecting the numerical columns of data and adding their names in a list 
numeric_columns = ['visitors', 'ad_impressions', 'views_trailer', 'views_content']
plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(cData[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

In [ ]:
# to find the 25th percentile and 75th percentile for the numerical columns.
Q1 = cData[numeric_columns].quantile(0.25)
Q3 = cData[numeric_columns].quantile(0.75)

IQR = Q3 - Q1                   #Inter Quantile Range (75th percentile - 25th percentile)

lower_whisker = Q1 - 1.5*IQR    #Finding lower and upper bounds for all values. All values outside these bounds are outliers
upper_whisker = Q3 + 1.5*IQR

In [ ]:
# Percentage of outliers in each column
((cData[numeric_columns] < lower_whisker) | (cData[numeric_columns] > upper_whisker)).sum()/cData.shape[0]*100

* There are quite a lot outliers in the data
* However, we will not treat them as they are proper values

### Data Preparation for Modeling

* We want to predict the first-day viewership.

In [ ]:
# defining X and y variables
X = cData.drop(["views_content"], axis=1)
y = cData["views_content"]

print(X.head())
print(y.head())

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

In [ ]:
# creating dummy variables
X = pd.get_dummies(
    X,
    columns=X.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
X.head()

In [ ]:
# converting the input attributes into float type for modeling
X = X.astype(float)
X.head()

In [ ]:
# splitting the data in 70:30 ratio for train to test data

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
print("Number of rows in train data =", x_train.shape[0])
print("Number of rows in test data =", x_test.shape[0])

### Model Building - Linear Regression

In [ ]:
olsmodel = sm.OLS(y_train, x_train).fit()
print(olsmodel.summary())

#### R-squared:
       * the value for R-squared is 0.792, which is good.
#### Adjusted. R-squared:
       * the value for adj. R-squared is 0.785, which is good.
#### const coefficient:
       * the value for const coefficient is 0.0602

### Model Performance Check

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
olsmodel_train_perf = model_performance_regression(olsmodel, x_train, y_train)
olsmodel_train_perf

**Observations**

- The training $R^2$ is 0.79, so the model is not underfitting

- The train and test RMSE and MAE are comparable, so the model is not overfitting either

- MAE suggests that the model can predict First day viewership within a mean error of 0.034 on the test data

- MAPE of 12.6 on the test data means that we are able to predict within 8.55% of the First day viewership

### Checking Linear Regression Assumptions

We will be checking the following Linear Regression assumptions:

1. **No Multicollinearity**

2. **Linearity of variables**

3. **Independence of error terms**

4. **Normality of error terms**

5. **No Heteroscedasticity**

### TEST FOR MULTICOLLINEARITY

* **General Rule of thumb**:
    - If VIF is between 1 and 5, then there is low multicollinearity.
    - If VIF is between 5 and 10, we say there is moderate multicollinearity.
    - If VIF is exceeding 10, it shows signs of high multicollinearity.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


def checking_vif(predictors):
    vif = pd.DataFrame()
    vif["feature"] = predictors.columns

    # calculating VIF for each feature
    vif["VIF"] = [
        variance_inflation_factor(predictors.values, i)
        for i in range(len(predictors.columns))
    ]
    return vif

In [ ]:
checking_vif(x_train)

* There are no multiple columns with very high VIF values, indicating no presence of multicollinearity
* We will systematically drop numerical columns with VIF > 5
* We will ignore the VIF values for dummy variables and the constant (intercept)

### Dealing with high p-value variables

- Some of the dummy variables in the data have p-value > 0.05. So, they are not significant and we'll drop them
- But sometimes p-values change after dropping a variable. So, we'll not drop all variables at once
- Instead, we will do the following:
    - Build a model, check the p-values of the variables, and drop the column with the highest p-value
    - Create a new model without the dropped feature, check the p-values of the variables, and drop the column with the highest p-value
    - Repeat the above two steps till there are no columns with p-value > 0.05

**Note**: The above process can also be done manually by picking one variable at a time that has a high p-value, dropping it, and building a model again. But that might be a little tedious and using a loop will be more efficient.

In [ ]:
# initial list of columns
predictors = x_train.copy()
cols = predictors.columns.tolist()

# setting an initial max p-value
max_p_value = 1

while len(cols) > 0:
    # defining the train set
    x_train_aux = predictors[cols]

    # fitting the model
    model = sm.OLS(y_train, x_train_aux).fit()

    # getting the p-values and the maximum p-value
    p_values = model.pvalues
    max_p_value = max(p_values)

    # name of the variable with maximum p-value
    feature_with_p_max = p_values.idxmax()

    if max_p_value > 0.05:
        cols.remove(feature_with_p_max)
    else:
        break

selected_features = cols
print(selected_features)

In [ ]:
x_train2 = x_train[selected_features]
x_test2 = x_test[selected_features]

In [ ]:
olsmod2 = sm.OLS(y_train, x_train2).fit()
print(olsmod2.summary())

In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
olsmod2_train_perf = model_performance_regression(olsmod2, x_train2, y_train)
olsmod2_train_perf

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
olsmod2_test_perf = model_performance_regression(olsmod2, x_test2, y_test)
olsmod2_test_perf

**Observations**

* Now no feature has p-value greater than 0.05, so we'll consider the features in *x_train2* as the final set of predictor variables and *olsmod2* as the final model to move forward with
* Now adjusted R-squared is 0.785, i.e., our model is able to explain ~79% of the variance
* The adjusted R-squared in *olsmod1* (where we considered the variables without multicollinearity) was 0.785
    * This shows that the variables we dropped were not affecting the model
* RMSE and MAE values are comparable for train and test sets, indicating that the model is not overfitting

**Now we'll check the rest of the assumptions on *olsmod2*.**

2. **Linearity of variables**

3. **Independence of error terms**

4. **Normality of error terms**

5. **No Heteroscedasticity**

In [ ]:
# let us create a dataframe with actual, fitted and residual values
df_pred = pd.DataFrame()

df_pred["Actual Values"] = y_train  # actual values
df_pred["Fitted Values"] = olsmod2.fittedvalues  # predicted values
df_pred["Residuals"] = olsmod2.resid  # residuals

df_pred.head()

In [ ]:
# let's plot the fitted values vs residuals

sns.residplot(
    data=df_pred, x="Fitted Values", y="Residuals", color="purple", lowess=True
)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted vs Residual plot")
plt.show()

* The scatter plot shows the distribution of residuals (errors) vs fitted values (predicted values).

* If there exist any pattern in this plot, we consider it as signs of non-linearity in the data and a pattern means that the model doesn't capture non-linear effects.

* **We see no pattern in the plot above. Hence, the assumptions of linearity and independence are satisfied.**

### TEST FOR NORMALITY

In [ ]:
sns.histplot(data=df_pred, x="Residuals", kde=True)
plt.title("Normality of residuals")
plt.show()

- The histogram of residuals does have a bell shape.
- Let's check the Q-Q plot.

In [ ]:
import pylab
import scipy.stats as stats

stats.probplot(df_pred["Residuals"], dist="norm", plot=pylab)
plt.show()

- The residuals more or less follow a straight line except for the tails.
- Let's check the results of the Shapiro-Wilk test.

In [ ]:
stats.shapiro(df_pred["Residuals"])

- Since p-value < 0.05, the residuals are not normal as per the Shapiro-Wilk test.
- Strictly speaking, the residuals are not normal.
- However, as an approximation, we can accept this distribution as close to being normal.
- **So, the assumption is satisfied.**

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip

name = ["F statistic", "p-value"]
test = sms.het_goldfeldquandt(df_pred["Residuals"], x_train2)
lzip(name, test)

**Since p-value > 0.05, we can say that the residuals are homoscedastic. So, this assumption is satisfied.**

## Predictions on test data

Now that we have checked all the assumptions of linear regression and they are satisfied, let's go ahead with prediction.

In [ ]:
# predictions on the test set
pred = olsmod2.predict(x_test2)

df_pred_test = pd.DataFrame({"Actual": y_test, "Predicted": pred})
df_pred_test.sample(10, random_state=1)

- We can observe here that our model has returned pretty good prediction results, and the actual and predicted values are comparable

## Final Model

In [ ]:
x_train_final = x_train2.copy()
x_test_final = x_test2.copy()

In [ ]:
olsmodel_final = sm.OLS(y_train, x_train_final).fit()
print(olsmodel_final.summary())

In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
olsmodel_final_train_perf = model_performance_regression(
    olsmodel_final, x_train_final, y_train
)
olsmodel_final_train_perf

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
olsmodel_final_test_perf = model_performance_regression(
    olsmodel_final, x_test_final, y_test
)
olsmodel_final_test_perf

* The model is able to explain ~79% of the variation in the data

* The train and test RMSE and MAE are low and comparable. So, our model is not suffering from overfitting

* The MAPE on the test set suggests we can predict within 9.2% of the First day viewership of content

* Hence, we can conclude the model *olsmodel_final* is good for prediction as well as inference purposes

####  the linear regression equation.

In [ ]:
# let's check the model parameters
olsmodel_final.params

In [ ]:
# Let us write the equation of linear regression
Equation = "views_content ="
print(Equation, end=" ")
for i in range(len(x_train_final.columns)):
    if i == 0:
        print(olsmodel_final.params[i], "+", end=" ")
    elif i != len(x_train_final.columns) - 1:
        print(
            olsmodel_final.params[i],
            "* (",
            x_train_final.columns[i],
            ")",
            "+",
            end="  ",
        )
    else:
        print(olsmodel_final.params[i], "* (", x_train_final.columns[i], ")")

### Conclusions and Recommendations

##### Significant Predictors:
1.	Visitors: Positive coefficient (0.1291) and a p-value of 0.000, indicating a significant positive impact on views_content.
2.	Major Sports Event: Negative coefficient (-0.0606) and a p-value of 0.000, indicating a significant negative impact on views_content.
3.	Views Trailer: Positive coefficient (0.0023) and a p-value of 0.000, indicating a significant positive impact on views_content.
4.	Day of the Week:
•	Monday, Saturday, Sunday, Thursday, and Wednesday have significant positive coefficients, indicating higher views on these days compared to the reference day.
5.	Season:
•	Spring, Summer, and Winter have significant positive coefficients, indicating higher views in these seasons compared to the reference s##### eason.
Recommendations:
6.	Increase Visitor Engagement:
•	Since the number of visitors has a significant positive impact on views_content, efforts should be made to increase visitor engagement through marketing and promotional activities.
7.	Leverage Trailers:
•	The positive impact of views_trailer suggests that promoting trailers can effectively increase content views. Consider investing in high-quality trailers and promoting them across various platforms.
8.	Optimize Content Release Schedule:
•	Given the significant positive impact of certain days of the week (Monday, Saturday, Sunday, Thursday, and Wednesday), consider scheduling major content releases on these days to maximize views.
9.	Consider Seasonal Trends:
•	The positive impact of Spring, Summer, and Winter seasons indicates that content views are higher during these periods. Plan major content releases and marketing campaigns around these seasons to capitalize on higher viewership.
10.	Monitor Major Sports Events:
•	The negative impact of major sports events suggests that viewership decreases during these times. Avoid scheduling major content releases during major sports events to prevent a drop in views.
